# 의견 말하기 유형 질문 데이터 만들기

- 의견 말하기 유형 질문 생성 만들기
- 질문에 대한 오디오 만들기


## 질문 생성 Chain

In [ ]:
import json
from typing import List

from tqdm import tqdm
from langchain_core.prompts import PromptTemplate, ChatPromptTemplate
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import JsonOutputParser, CommaSeparatedListOutputParser
from pydantic import BaseModel, Field
# from langchain.schema import HumanMessage, AIMessage, StrOutputParser
from langchain_core.messages import HumanMessage, AIMessage
from langchain_core.output_parsers import StrOutputParser
import pandas as pd


import os


In [12]:
model = ChatOpenAI(model="gpt-5.6-luna")

### 질문 주제 샘플링하기

In [13]:
csv_parser = CommaSeparatedListOutputParser()

In [14]:
csv_format_instruction = csv_parser.get_format_instructions()

In [15]:
subjet_prompt_template = PromptTemplate.from_template(template="영어 시험에 나올 법한 일상적인 주제를 단어 형식으로 만들어줘.\n{format_instruction}",
                                                      partial_variables={"format_instruction": csv_format_instruction})

In [16]:
subject_chain = subjet_prompt_template | model | csv_parser

In [17]:
subject_list = subject_chain.invoke({})
subject_list

['family',
 'school',
 'hobbies',
 'friendship',
 'food',
 'health',
 'exercise',
 'travel',
 'weather',
 'shopping',
 'transportation',
 'technology',
 'environment',
 'jobs',
 'holidays',
 'daily routines',
 'communication',
 'culture',
 'sports',
 'community']

In [18]:
subject_list = subject_list[:4]
subject_list

['family', 'school', 'hobbies', 'friendship']

### 질문 만들기

In [19]:
template = """\
- 영어 스피킹 시험 중에 의견 말하기(express an opinion)에 나올 법한 {input} 주제 관련 질문 하나 만들어줘
- 상대방에게 그렇게 생각한 이유와 주장을 이야기하도록 요구해줘
- 한 문장만 만들어줘
- 여러 예시 만들지마
- 영어로
- 예시
Some people prefer to take a job that does not pay well but does provide a lot of time off from work. What is your opinion about taking a job with a low salary that has a lot of vacation time? Give reasons for your opinion.
"""

question_prompt_template = PromptTemplate.from_template(template=template)

In [20]:
question_chain = question_prompt_template | model | StrOutputParser()

In [21]:
question_list = []
for subject in tqdm(subject_list):
    question_list.append(question_chain.invoke({"input": subject}))

100%|██████████| 4/4 [00:07<00:00,  1.87s/it]


In [22]:
question_list

['Some people believe that families should eat dinner together every day, while others prefer flexible meal times; what is your opinion, and what reasons support your view?',
 'Some people believe that schools should give students less homework and more free time, while others think regular homework is essential for learning; what is your opinion, and what reasons and arguments support your view?',
 'Some people prefer hobbies they can enjoy alone, while others prefer hobbies they can do with friends; what is your opinion, and what reasons support your view?',
 'Some people believe that having many friends is more important than having a few close friends; what is your opinion, and what reasons and arguments support your view?']

## 질문에 대한 오디오 파일 만들기

In [23]:
from openai import OpenAI

In [24]:
client = OpenAI()

In [25]:
def gen_speech_file(text, output_file_path):
    response = client.audio.speech.create(
        model="tts-1",
        voice="alloy", # alloy, echo, fable, onyx, nova, and shimmer
        input=text
    )
    response.stream_to_file(output_file_path)

In [26]:
!mkdir -p ./data/speaking__express_an_opinion

���� ������ �ùٸ��� �ʽ��ϴ�.


In [27]:
save_dir = "./data/speaking__express_an_opinion"

In [28]:
question_list

['Some people believe that families should eat dinner together every day, while others prefer flexible meal times; what is your opinion, and what reasons support your view?',
 'Some people believe that schools should give students less homework and more free time, while others think regular homework is essential for learning; what is your opinion, and what reasons and arguments support your view?',
 'Some people prefer hobbies they can enjoy alone, while others prefer hobbies they can do with friends; what is your opinion, and what reasons support your view?',
 'Some people believe that having many friends is more important than having a few close friends; what is your opinion, and what reasons and arguments support your view?']

In [29]:
record_list = []

for i, q in tqdm(enumerate(question_list), total=len(question_list)):
    output_file_path = f"{save_dir}/question_{i}.wav"
    gen_speech_file(q, output_file_path)

    record = {"question": q, "audio_file_path": output_file_path}
    record_list.append(record)

  0%|          | 0/4 [00:00<?, ?it/s]C:\Users\jaeyo\AppData\Local\Temp\ipykernel_33756\455120697.py:7: DeprecationWarning: Due to a bug, this method doesn't actually stream the response content, `.with_streaming_response.method()` should be used instead
  response.stream_to_file(output_file_path)
100%|██████████| 4/4 [00:10<00:00,  2.73s/it]


In [30]:
df = pd.DataFrame(record_list)
df

,question,audio_file_path
0,Some people believe that families should eat d...,./data/speaking__express_an_opinion/question_0...
1,Some people believe that schools should give s...,./data/speaking__express_an_opinion/question_1...
2,Some people prefer hobbies they can enjoy alon...,./data/speaking__express_an_opinion/question_2...
3,Some people believe that having many friends i...,./data/speaking__express_an_opinion/question_3...


In [31]:
df.to_csv(f"{save_dir}/question_and_audio.csv", index=False)

In [32]:
from IPython.display import Audio

In [33]:
Audio(f"{save_dir}/question_2.wav")

In [34]:
subject_list

['family', 'school', 'hobbies', 'friendship']